# Phase 1 MulT (Multimodal Transformer) on CMU-MOSEI

Notebook này hỗ trợ profile `colab`, đọc dữ liệu từ Google Drive/GCS và huấn luyện mô hình **MulT** (Multimodal Transformer) với Cross-Modal Attention, Attention Pooling, và Enhanced Fusion Head.

Mục tiêu: Đạt Test MAE ≤ 0.5700, Test Corr ≥ 0.7300 (vượt Improved LSTM).

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print('Not running inside Google Colab. Mount step skipped.')

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import sys
from pathlib import Path

RUNTIME_PROFILE = 'colab'
REPO_SOURCE = 'git'
REPO_URL = 'https://github.com/Kandesfx/Training-Multimodal-Emotion-Analysis.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCDA')
REPO_PATH = Path('/content/BCDA')
DATA_PKL_OVERRIDE = None
USE_DRIVE_OUTPUTS = True
RESUME_TRAINING = False
RESUME_CHECKPOINT_TYPE = 'last'
BEST_CHECKPOINT_NAME = 'best_model_mult.pt'
LAST_CHECKPOINT_NAME = 'last_model_mult.pt'
USE_GCS = True
GCS_BUCKET = 'mer-data-bucket-kandesfx'
WANDB_ENABLE = True
WANDB_PROJECT = 'bcda-phase1'

if RUNTIME_PROFILE != 'colab':
    raise ValueError('Notebook này hiện được tối ưu cho profile colab.')

if REPO_SOURCE not in {'git', 'drive'}:
    raise ValueError('REPO_SOURCE must be either \'git\' or \'drive\'.')

if REPO_SOURCE == 'git':
    if not REPO_PATH.exists():
        if '<YOUR_REPO_URL_HERE>' in REPO_URL:
            raise ValueError('Hãy thay REPO_URL bằng URL repo thật trước khi chạy cell này.')
        get_ipython().system(f'git clone {REPO_URL} {REPO_PATH}')
    else:
        print(f'Repo already exists at {REPO_PATH}')
else:
    REPO_PATH = DRIVE_ROOT

%cd {REPO_PATH}
if str(REPO_PATH) not in sys.path:
    sys.path.append(str(REPO_PATH))

!python -m pip install -q --upgrade pip
!python -m pip install -q torch torchvision torchaudio numpy pandas scikit-learn matplotlib seaborn tqdm wandb

if IN_COLAB and USE_GCS:
    from google.colab import auth
    print('Authenticating for GCS access...')
    auth.authenticate_user()
    print('Downloading aligned_50.pkl from GCS...')
    get_ipython().system(f'mkdir -p /content/data/MSA-Dataset')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/data/MSA-Dataset/aligned_50.pkl /content/data/MSA-Dataset/aligned_50.pkl')

In [ ]:
from training.config_phase1 import Phase1Config
from training.dataset_mosei import create_dataloaders
from training.trainer import Phase1Trainer
from training.models.mult import MulTRegressor

# Build config
config = Phase1Config()
config.model_type = 'mult'
config.runtime.use_drive_outputs_on_colab = USE_DRIVE_OUTPUTS
config.runtime.use_gcs = USE_GCS
config.runtime.gcs_bucket = GCS_BUCKET
config.wandb.enable = WANDB_ENABLE
config.wandb.project = WANDB_PROJECT
config.apply_profile('colab', drive_root=DRIVE_ROOT, repo_root=REPO_PATH)

if DATA_PKL_OVERRIDE is not None:
    config.override_paths(mosei_pkl=DATA_PKL_OVERRIDE)

# === MulT-specific hyperparameters ===
config.mult_model.d_model = 64
config.mult_model.num_heads = 4
config.mult_model.num_cross_layers = 4
config.mult_model.num_self_layers = 2
config.mult_model.ffn_dim = 128
config.mult_model.attn_dropout = 0.1
config.mult_model.fusion_hidden_dim = 128
config.mult_model.fusion_dropout = 0.3

# === Training hyperparameters (Transformer-optimized) ===
config.training.batch_size = 32
config.training.num_workers = 2
config.training.num_epochs = 50
config.training.patience = 10
config.training.scheduler_patience = 4
config.training.learning_rate = 5e-4      # Lower than LSTM (1e-3)
config.training.weight_decay = 1e-3       # Higher regularization for Transformers
config.training.max_grad_norm = 0.5       # Lower clip for Transformers
config.training.use_amp = True
config.training.resume_from_checkpoint = RESUME_TRAINING
config.training.resume_checkpoint_type = RESUME_CHECKPOINT_TYPE
config.training.checkpoint_name = BEST_CHECKPOINT_NAME
config.training.last_checkpoint_name = LAST_CHECKPOINT_NAME
config.setup()

if USE_GCS and RESUME_TRAINING:
    print('Checking GCS for existing checkpoints to resume...')
    get_ipython().system(f'mkdir -p {config.paths.checkpoints_dir}')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/checkpoints/phase1/{BEST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{BEST_CHECKPOINT_NAME} || true')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/checkpoints/phase1/{LAST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{LAST_CHECKPOINT_NAME} || true')
    get_ipython().system(f'mkdir -p {config.paths.logs_dir}')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/logs/phase1/history.csv {config.paths.logs_dir}/history.csv || true')

if WANDB_ENABLE:
    import wandb
    wandb.login()

print('Runtime profile:', config.runtime.profile)
print('Model type:', config.model_type)
print('Resume training:', config.training.resume_from_checkpoint)
print('Resume checkpoint type:', config.training.resume_checkpoint_type)
print('Best checkpoint name:', config.training.checkpoint_name)
print('Last checkpoint name:', config.training.last_checkpoint_name)
print()
print('=== MulT Model Config ===')
print(f'd_model: {config.mult_model.d_model}')
print(f'num_heads: {config.mult_model.num_heads}')
print(f'num_cross_layers: {config.mult_model.num_cross_layers}')
print(f'num_self_layers: {config.mult_model.num_self_layers}')
print(f'ffn_dim: {config.mult_model.ffn_dim}')
print(f'attn_dropout: {config.mult_model.attn_dropout}')
print(f'fusion_hidden_dim: {config.mult_model.fusion_hidden_dim}')
print(f'fusion_dropout: {config.mult_model.fusion_dropout}')
print()
print('=== Training Config ===')
print(f'learning_rate: {config.training.learning_rate}')
print(f'weight_decay: {config.training.weight_decay}')
print(f'max_grad_norm: {config.training.max_grad_norm}')
print(f'patience: {config.training.patience}')
for key, value in config.paths.as_dict().items():
    print(f'{key}: {value}')

In [ ]:
required_paths = [
    config.paths.mosei_pkl,
    config.paths.checkpoints_dir,
    config.paths.logs_dir,
    config.paths.outputs_dir,
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:
' + '
'.join(missing))

best_checkpoint_path = config.paths.checkpoints_dir / config.training.checkpoint_name
last_checkpoint_path = config.paths.checkpoints_dir / config.training.last_checkpoint_name

print(f'Best checkpoint path: {best_checkpoint_path}')
print(f'Last checkpoint path: {last_checkpoint_path}')
print(f'Best checkpoint exists: {best_checkpoint_path.exists()}')
print(f'Last checkpoint exists: {last_checkpoint_path.exists()}')
print('All required paths are ready.')

In [ ]:
# Smoke test: Verify MulT model forward and backward pass
import torch

print('=== Smoke Test: MulTRegressor ===')
model = MulTRegressor(config.mult_model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Create dummy input
B, S = 4, 50
text = torch.randn(B, S, 768)
audio = torch.randn(B, S, 74)
vision = torch.randn(B, S, 35)

# Forward pass
output = model(text=text, audio=audio, vision=vision)
assert output.shape == (B,), f'Expected shape ({B},), got {output.shape}'
print(f'Forward pass OK. Output shape: {output.shape}')

# Backward pass
loss = output.sum()
loss.backward()
print('Backward pass OK.')
print('All smoke tests passed!')

In [ ]:
dataloaders = create_dataloaders(config=config, pkl_path=config.paths.mosei_pkl)
sample_batch = next(iter(dataloaders['train']))

for key, value in sample_batch.items():
    if hasattr(value, 'shape'):
        print(key, value.shape, value.dtype)
    else:
        print(key, type(value), value[:2] if isinstance(value, list) else value)

print()
print('Dataset sizes:')
for split, loader in dataloaders.items():
    print(f'  {split}: {len(loader.dataset)} samples, {len(loader)} batches')

In [ ]:
# Instantiate model and start training
model = MulTRegressor(config.mult_model)
trainer = Phase1Trainer(model=model, config=config)
summary = trainer.fit(dataloaders['train'], dataloaders['valid'])
test_metrics = trainer.evaluate_and_save(dataloaders['test'], split='test', epoch=summary['best_epoch'])
summary, test_metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = config.paths.logs_dir / 'history.csv'
history_df = pd.read_csv(history_path)
history_df.tail(20)

In [ ]:
valid_df = history_df[history_df['split'] == 'valid'].copy()
train_df = history_df[history_df['split'] == 'train'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(train_df['epoch'], train_df['loss'], label='train_loss', marker='o', markersize=3)
if 'train_step_loss' in train_df.columns:
    axes[0].plot(train_df['epoch'], train_df['train_step_loss'], label='train_step_loss', linestyle='--', alpha=0.7)
axes[0].plot(valid_df['epoch'], valid_df['loss'], label='valid_loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation metrics
axes[1].plot(valid_df['epoch'], valid_df['mae'], label='valid_mae', marker='o', markersize=3, color='tab:orange')
axes[1].plot(valid_df['epoch'], valid_df['corr'], label='valid_corr', marker='s', markersize=3, color='tab:green')
axes[1].axhline(y=0.5700, color='r', linestyle='--', label='Target MAE (0.5700)', alpha=0.7)
axes[1].axhline(y=0.7300, color='g', linestyle='--', label='Target Corr (0.7300)', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric Value')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Overfitting ratio (train/valid loss)
merged = train_df.merge(valid_df, on='epoch', suffixes=('_train', '_valid'))
overfit_ratio = merged['loss_train'] / merged['loss_valid']
axes[2].plot(merged['epoch'], overfit_ratio, label='Train/Valid Loss Ratio', marker='d', markersize=3, color='tab:purple')
axes[2].axhline(y=1.0, color='k', linestyle='--', label='Perfect (1.0)', alpha=0.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Overfitting Ratio')
axes[2].set_title('Overfitting Ratio (Train/Valid Loss)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare with LSTM baselines and MMSA benchmark
print('=== Test Results Comparison ===')
print()
print(f'{"Model":<25} {"MAE":>8} {"Corr":>8} {"Acc-2":>8} {"Acc-5":>8} {"Acc-7":>8}')
print('-' * 75)
print(f'{"Baseline LSTM":<25} {"0.6071":>8} {"0.6995":>8} {"81.03%":>8} {"49.73%":>8} {"48.36%":>8}')
print(f'{"Improved LSTM":<25} {"0.5859":>8} {"0.7229":>8} {"81.37%":>8} {"51.53%":>8} {"49.71%":>8}')
print(f'{"MMSA Benchmark MulT":<25} {"0.5593":>8} {"0.7331":>8} {"81.15%":>8} {"54.18%":>8} {"52.84%":>8}')
print('-' * 75)

test_row = history_df[history_df['split'] == 'test'].tail(1)
if len(test_row) > 0:
    mae = test_row['mae'].values[0]
    corr = test_row['corr'].values[0]
    acc2 = test_row['acc2'].values[0]
    acc5 = test_row['acc5'].values[0]
    acc7 = test_row['acc7'].values[0]
    print(f'{"MulT (Ours)":<25} {mae:>8.4f} {corr:>8.4f} {acc2:>8.2f}% {acc5:>8.2f}% {acc7:>8.2f}%')
    print()
    # Improvement over Improved LSTM
    mae_imp = 0.5859 - mae
    corr_imp = corr - 0.7229
    print(f'Improvement over Improved LSTM:')
    mae_diff_str = f"-{mae_imp:.4f}" if mae_imp >= 0 else f"+{-mae_imp:.4f}"
    mae_status = "↓ better" if mae_imp > 0 else "↑ worse"
    print(f'  MAE:  {mae_diff_str} ({mae_status})')
    corr_diff_str = f"+{corr_imp:.4f}" if corr_imp >= 0 else f"{corr_imp:.4f}"
    corr_status = "↑ better" if corr_imp > 0 else "↓ worse"
    print(f'  Corr: {corr_diff_str} ({corr_status})')
    print()
    print(f'Target: MAE ≤ 0.5700, Corr ≥ 0.7300')
    print(f'Status:  MAE {"✓ PASS" if mae <= 0.5700 else "✗ FAIL"}, Corr {"✓ PASS" if corr >= 0.7300 else "✗ FAIL"}')